# AnimationStudio — Pre-flight validation (one clear image per step)

Run this notebook **before** the full generation notebook. It walks each stage
of the stack and produces a single clear test image so you can confirm the
pipeline works end to end — without burning hours on a full Phase-1 run.

Steps:

1. GPU / VRAM available + studio installed
2. ComfyUI installed
3. Flux model downloaded to the Colab disk (colab-gpu fp8 bundle)
4. ComfyUI server started and fully answering the API
5. The model is visible to ComfyUI (right filename, right loader)
6. Backend + generation input built against the repo engine
7. **One** 1024×1024 test image is generated, saved to the repo, and shown
   inline
8. Automatic sharpness check on that image

**Run top to bottom** (Runtime → Run all). If the Colab VM/kernel restarts
mid-run, **re-run from Cell 1 (Settings)** — every step from STEP 4 onward is
self-healing and will restart ComfyUI itself, but earlier variables are wiped
by the restart.

If step 8 shows a sharp, visible image → proceed to the main notebook.
If it shows a flat/blurry image → stop; the troubleshooting list at the end
applies.


In [ ]:
#@title 0. Settings

import os
import subprocess
import sys
import time
import shutil
from pathlib import Path

REPO_URL = "https://github.com/YOUR_ORG/AnimationStudio.git"  #@param {type:"string"}
BRANCH = "colab-gpu"  #@param ["colab-gpu"]
# master is deprecated: its encoder/VAE URLs return 404. All runs use colab-gpu.

COMFYUI_PORT = 8188  #@param {type:"integer"}
TEST_SIZE = 1024  #@param {type:"integer"}
TEST_SEED = 42  #@param {type:"integer"}
TEST_PROMPT = "Lily Bunny, cute anthropomorphic white rabbit child, fluffy fur, big round expressive eyes, soft studio lighting, crisp clean 3D render, bright cheerful colors, high detail, sharp focus"  #@param {type:"string"}
TEST_NEGATIVE = "blurry, out of focus, low quality, deformed, distorted, text, watermark, logo"  #@param {type:"string"}

# N-08: fail fast when free disk cannot hold the ~17.25 GB fp8 model.
MIN_FREE_GB = 25  #@param {type:"integer"}

WORK = "/content"
REPO = f"{WORK}/AnimationStudio"
COMFY = f"{WORK}/comfyui"

# N-10/N-11: the colab-gpu model is fp8 Flux dev (~17.25 GB), no longer ~12 GB.
MODEL_FILE = "flux1-dev.safetensors"  # the name the ComfyUI loader must see
MODEL_URL = "https://huggingface.co/Comfy-Org/flux1-dev/resolve/main/flux1-dev-fp8.safetensors"
MODEL_EXPECTED_BYTES = 17250000000  # ~17.25 GB (decimal)

if REPO_URL.startswith("https://github.com/YOUR_ORG/"):
    raise SystemExit("Set REPO_URL in Cell 1 before running.")


def run(cmd, **kw):
    print("+ " + " ".join(cmd))
    return subprocess.run(cmd, check=True, **kw)


_disk = shutil.disk_usage(WORK)
free_gb = _disk.free / 1e9
print(f"Free disk on {WORK}: {free_gb:.1f} GB (need >= {MIN_FREE_GB} GB)")
if free_gb < MIN_FREE_GB:
    raise SystemExit(
        f"Only {free_gb:.1f} GB free — the ~17.25 GB fp8 download will not fit. "
        "Free space on the VM or lower MIN_FREE_GB."
    )


In [ ]:
#@title 1. STEP 1 — Clone repo, install studio, check GPU/VRAM

os.chdir(WORK)
if not os.path.isdir(REPO):
    run(["git", "clone", "--branch", BRANCH, REPO_URL, "AnimationStudio"])
os.chdir(REPO)
run(["git", "checkout", BRANCH])
run(["git", "pull", "origin", BRANCH])

run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
run([sys.executable, "-m", "pip", "install", "-q", "requests", "pillow", "numpy", "IPython"])

import torch
assert torch.cuda.is_available(), "CUDA not available on this runtime"
dev = torch.cuda.get_device_properties(0)
assert dev.major >= 7, (
    f"GPU compute capability {dev.major}.{dev.minor} < 7.0 — fp8 Flux will not run."
)
print("PASS: GPU =", torch.cuda.get_device_name(0))
print("      VRAM =", round(dev.total_memory / 1e9, 1), "GB")


In [ ]:
#@title 2. STEP 2 — Install ComfyUI

if not os.path.isdir(COMFY):
    run(["git", "clone", "--depth", "1",
         "https://github.com/comfyanonymous/ComfyUI.git", COMFY])
run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{COMFY}/requirements.txt"])
print("PASS: ComfyUI installed at", COMFY)


In [ ]:
#@title 3. STEP 3 — Download the Flux model (Colab disk, symlink into ComfyUI)

# Mirrors the Phase-1 notebook (colab-gpu branch): wget -c into an ephemeral
# cache, then symlink into ComfyUI's checkpoints dir so the loader sees
# "flux1-dev.safetensors".  Re-run after a VM reset (wget -c resumes).

cache = f"{WORK}/models/checkpoints/{MODEL_FILE}"
link = f"{COMFY}/models/checkpoints/{MODEL_FILE}"

if not (os.path.exists(cache) and os.path.getsize(cache) > 0):
    os.makedirs(os.path.dirname(cache), exist_ok=True)
    print("Downloading fp8 Flux dev (~17.25 GB) ...")
    run(["wget", "-q", "-c", "-O", cache, MODEL_URL])

size = os.path.getsize(cache)
print(f"cached: {size / 1e9:.2f} GB")
assert size > 0, "Downloaded model is empty"
assert size >= MODEL_EXPECTED_BYTES * 0.9, (
    f"Model download looks truncated: {size / 1e9:.2f} GB < 90% of "
    f"{MODEL_EXPECTED_BYTES / 1e9:.2f} GB"
)

os.makedirs(os.path.dirname(link), exist_ok=True)
if os.path.lexists(link) and not os.path.islink(link):
    os.remove(link)
if not os.path.islink(link):
    try:
        os.symlink(cache, link)
    except OSError:
        shutil.copyfile(cache, link)
print("PASS: model downloaded and linked for ComfyUI as", MODEL_FILE)


In [ ]:
#@title 4. STEP 4 — Start ComfyUI (via the shared helper)

# ensure_comfyui_up() is imported from colab/comfy_helpers.py so later cells
# can auto-restart the server after a Colab VM recycle instead of failing
# with Connection refused.  It also survives kernel restarts.

REPO = globals().get("REPO", "/content/AnimationStudio")
COMFY = globals().get("COMFY", "/content/comfyui")
WORK = globals().get("WORK", "/content")
COMFYUI_PORT = globals().get("COMFYUI_PORT", 8188)

import sys
sys.path.insert(0, f"{REPO}/colab")
from comfy_helpers import ensure_comfyui_up, comfy_alive, server_url  # noqa

ensure_comfyui_up(port=COMFYUI_PORT, work=WORK, comfy_dir=COMFY)


In [ ]:
#@title 5. STEP 5 — Verify the model is visible to ComfyUI

# Self-healing: imports + defaults restore state after a VM restart.

REPO = globals().get("REPO", "/content/AnimationStudio")
COMFY = globals().get("COMFY", "/content/comfyui")
WORK = globals().get("WORK", "/content")
COMFYUI_PORT = globals().get("COMFYUI_PORT", 8188)
MODEL_FILE = globals().get("MODEL_FILE", "flux1-dev.safetensors")

import os
import sys
import requests
sys.path.insert(0, f"{REPO}/colab")
from comfy_helpers import ensure_comfyui_up

ensure_comfyui_up(port=COMFYUI_PORT, work=WORK, comfy_dir=COMFY)

try:
    info = requests.get(f"http://127.0.0.1:{COMFYUI_PORT}/object_info", timeout=60).json()
except Exception as exc:
    raise SystemExit(f"ComfyUI object_info request failed: {exc}")


def _options(node, key):
    try:
        return sorted(list(info[node]["input"]["required"][key][0]))
    except Exception:
        return []


checkpoints = _options("CheckpointLoaderSimple", "ckpt_name")
unets = _options("UnetLoaderGGUF", "unet_name")  # [] when ComfyUI-GGUF absent

print("checkpoints:", checkpoints)
print("gguf unets: ", unets)

visible = MODEL_FILE in checkpoints or MODEL_FILE in unets
print("PASS: model visible to ComfyUI" if visible else
      "FAIL: model file not found by ComfyUI — re-run STEP 3, then the loader check")
assert visible, f"Expected to see {MODEL_FILE} in the loader lists above"


In [ ]:
#@title 6. STEP 6 — Build the generation backend + input

# Self-healing: re-imports and re-derives state so this cell also works after
# a VM restart once Settings + STEP 4 have run.

REPO = globals().get("REPO", "/content/AnimationStudio")
COMFY = globals().get("COMFY", "/content/comfyui")
WORK = globals().get("WORK", "/content")
COMFYUI_PORT = globals().get("COMFYUI_PORT", 8188)
TEST_PROMPT = globals().get(
    "TEST_PROMPT",
    "Lily Bunny, cute anthropomorphic white rabbit child, fluffy fur, big round expressive eyes, "
    "soft studio lighting, crisp clean 3D render, bright cheerful colors, high detail, sharp focus",
)
TEST_NEGATIVE = globals().get(
    "TEST_NEGATIVE",
    "blurry, out of focus, low quality, deformed, distorted, text, watermark, logo",
)
TEST_SIZE = globals().get("TEST_SIZE", 1024)
TEST_SEED = globals().get("TEST_SEED", 42)

import sys
sys.path.insert(0, REPO)
sys.path.insert(0, f"{REPO}/colab")
from comfy_helpers import ensure_comfyui_up
from src.generation_engine.base import GenerationInput
from src.generation_engine.comfy_backend import ComfyUIBackend

ensure_comfyui_up(port=COMFYUI_PORT, work=WORK, comfy_dir=COMFY)

backend = ComfyUIBackend(server_url=f"http://127.0.0.1:{COMFYUI_PORT}")
gen_input = GenerationInput(
    prompt=TEST_PROMPT,
    negative_prompt=TEST_NEGATIVE,
    seed=TEST_SEED,
    width=TEST_SIZE,
    height=TEST_SIZE,
    num_images=1,
)
print("PASS: backend =", type(backend).__name__, "| input size =", TEST_SIZE)


In [ ]:
#@title 7. STEP 7 — Generate ONE clear test image (via ComfyUI)

REPO = globals().get("REPO", "/content/AnimationStudio")
COMFY = globals().get("COMFY", "/content/comfyui")
WORK = globals().get("WORK", "/content")
COMFYUI_PORT = globals().get("COMFYUI_PORT", 8188)
TEST_SEED = globals().get("TEST_SEED", 42)

import sys
sys.path.insert(0, f"{REPO}/colab")
from comfy_helpers import ensure_comfyui_up

ensure_comfyui_up(port=COMFYUI_PORT, work=WORK, comfy_dir=COMFY)

try:
    out = backend.generate(gen_input, asset_type="")
except NameError as exc:
    raise SystemExit(
        "backend/gen_input not defined — run Cells 1-7 in order (a VM restart "
        "wipes earlier cell state)."
    ) from exc
assert out.images, f"No image returned: {out.metadata}"

img = out.images[0]
dst_dir = Path(REPO) / "Universe" / "_validate"
dst_dir.mkdir(parents=True, exist_ok=True)
dst = dst_dir / f"smoke_{TEST_SEED}.png"
img.save(dst, format="PNG")
print("saved:", dst)
print("size :", img.size, "|", f"{dst.stat().st_size / 1024:.1f} KB")

from IPython.display import Image as IPImage, display
display(IPImage(filename=str(dst)))
print("PASS: image generated and saved")


In [ ]:
#@title 8. STEP 8 — Automatic sharpness check (blur detection)

import os

dst = globals().get("dst")
assert dst is not None and os.path.isfile(dst), "STEP 7 has not produced an image — run it first"

import numpy as np
from PIL import Image as PILImage, ImageFilter

gray = PILImage.open(dst).convert("L")
edges = gray.filter(ImageFilter.FIND_EDGES)
score = float(np.asarray(edges, dtype=np.float32).var())

if score > 60:
    verdict = "SHARP — clear, detailed"
elif score > 15:
    verdict = "CHECK — some detail, decide visually above"
else:
    verdict = "BLURRY / FLAT — do NOT proceed"

print("edge-variance sharpness score:", round(score, 1))
print("verdict:", verdict)
print()
print("Compare: a solid-color placeholder (mock backend) scores ~0-50.")
print("A clear Flux image usually scores hundreds to thousands.")


## Next steps

- **Sharp and clear above?** → Run the main notebook
  (`AnimationStudio_Colab.ipynb`) Cells 9–11 for the full Phase-1 run. The
  workflows it uses now carry the same Flux-correct settings this notebook
  verified.
- **BLURRY / no image?** → check, in order:
  1. `comfyui.log` tail: `!tail -n 40 /content/comfyui.log`
  2. Cell 4 (STEP 3): model file size ≈17.25 GB and the symlink exists
  3. Cell 6 (STEP 5): the model name appears in the loader list
  4. VRAM: 16 GB T4 is tight for fp8 Flux — close other notebooks/VMs
  5. The old `mock:` placeholder PNGs in `Universe/...` are *expected*; newly
     generated files overwrite them via `--persist-images`.
